## Modelo Machine Learning

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

df = pd.read_csv('wfh_burnout_dataset.csv')


features = ['work_hours', 'screen_time_hours', 'meetings_count', 'breaks_taken',
            'after_hours_work', 'app_switches', 'sleep_hours', 'task_completion',
            'isolation_index', 'fatigue_score']
X = df[features]
y = df['burnout_risk']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.30, random_state=42, stratify=y_encoded)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print("Datos listos.")

Datos listos.


In [ ]:
# Árbol de decisión
modelo_tree = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
modelo_tree.fit(X_train, y_train)

num_parametros = modelo_tree.tree_.node_count
print(f"Número de parámetros: {num_parametros}")

def evaluar_tree(X, y_true, nombre):
    y_pred = modelo_tree.predict(X)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    print(f"--- {nombre} ---")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1-Score: {f1:.4f}\n")
    return acc, f1

evaluar_tree(X_train, y_train, "TRAIN")
evaluar_tree(X_val, y_val, "VALIDACIÓN")
evaluar_tree(X_test, y_test, "TEST")

print(classification_report(y_test, modelo_tree.predict(X_test), target_names=le.classes_))

Número de parámetros: 57
--- TRAIN ---
Accuracy: 0.9336
F1-Score: 0.8930

--- VALIDACIÓN ---
Accuracy: 0.8833
F1-Score: 0.8225

--- TEST ---
Accuracy: 0.9033
F1-Score: 0.8577

              precision    recall  f1-score   support

        High       0.61      0.95      0.74        21
         Low       0.97      0.93      0.95       153
      Medium       0.90      0.87      0.88       126

    accuracy                           0.90       300
   macro avg       0.83      0.92      0.86       300
weighted avg       0.92      0.90      0.91       300



In [ ]:
 # Crear el modelo
# Usamos 100 árboles y una profundidad máxima de 10 para no pasarnos
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

# Calcular "Parámetros" (Total de nodos en el bosque)
total_nodos = sum([arbol.tree_.node_count for arbol in rf_model.estimators_])

print(f"🌲 Número de árboles: 100")
print(f"📊 Número total de parámetros (nodos): {total_nodos}")

🌲 Número de árboles: 100
📊 Número total de parámetros (nodos): 16592


In [ ]:
def evaluar_ml(X, y_true, nombre):
    y_pred = rf_model.predict(X)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    print(f"--- {nombre} ---")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1-Score: {f1:.4f}\n")
    return acc, f1

evaluar_ml(X_train, y_train, "TRAIN")
evaluar_ml(X_val, y_val, "VALIDACIÓN")
evaluar_ml(X_test, y_test, "TEST")

print("INFORME DE TEST:")
print(classification_report(y_test, rf_model.predict(X_test), target_names=le.classes_))

--- TRAIN ---
Accuracy: 1.0000
F1-Score: 1.0000

--- VALIDACIÓN ---
Accuracy: 0.9767
F1-Score: 0.9543

--- TEST ---
Accuracy: 0.9633
F1-Score: 0.9085

INFORME DE TEST:
              precision    recall  f1-score   support

        High       0.93      0.67      0.78        21
         Low       0.99      0.99      0.99       153
      Medium       0.93      0.98      0.96       126

    accuracy                           0.96       300
   macro avg       0.95      0.88      0.91       300
weighted avg       0.96      0.96      0.96       300



### Conclusión final
En lugar de utilizar un Random Forest muy grande y complejo (con 16.592 nodos), hemos elegido un Árbol de Decisión mucho más sencillo, con solo 57 nodos. Aunque la precisión global es algo menor (90,33%), este modelo tiene una ventaja muy importante desde el punto de vista de la salud laboral: consigue un Recall de 0,95 en la clase “High”.

Esto quiere decir que identifica correctamente al 95% de los empleados que están en una situación de riesgo alto. En un sistema de prevención, es más importante detectar a casi todas las personas en situación crítica, aunque eso suponga perder algo de precisión general. En este contexto, es preferible asegurar que nadie en alto riesgo pase desapercibido